# Hybrid Retrieval Pipeline
**BM25 + Semantic Search (ChromaDB) + Cross-Encoder Reranker**

Pipeline:
```
Query → [BM25 top-50] ─┐
                        ├─ Reciprocal Rank Fusion → top-20 → Reranker → top-k
Query → [Dense  top-50] ┘
```

**Chạy local** (CPU, không cần GPU). Reranker sẽ tự dùng GPU nếu có.

Cấu trúc thư mục cần:
```
project/
├── dataset/
│   ├── chunks/
│   │   ├── legal_chunks.parquet
│   │   ├── forms_chunks.parquet
│   │   └── examples_chunks.parquet
│   ├── chromadb/          ← giải nén từ chromadb_output.zip
│   └── bm25/              ← tự tạo khi chạy notebook này
│       ├── bm25_legal.pkl
│       ├── bm25_forms.pkl
│       └── bm25_examples.pkl
└── models/
    ├── embedding/         ← cache multilingual-e5-large-instruct
    └── reranker/          ← cache bge-reranker-v2-m3
```

## 0. Cài đặt thư viện

In [ ]:
# Chạy cell này lần đầu
# !pip3 install rank-bm25 sentence-transformers chromadb underthesea unicodedata2 tqdm pandas pyarrow

## 1. Imports & Config

In [ ]:
import os
import re
import json
import time
import pickle
import unicodedata
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Imports OK")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ============================================================
# CONFIG — chỉnh sửa paths nếu khác cấu trúc thư mục
# ============================================================

PROJECT_ROOT   = Path(".")                          # thư mục gốc project
CHUNK_DIR      = PROJECT_ROOT / "dataset" / "chunks"
CHROMA_DIR     = PROJECT_ROOT / "dataset" / "chromadb"
BM25_DIR       = PROJECT_ROOT / "dataset" / "bm25"
MODEL_EMBED    = PROJECT_ROOT / "models" / "embedding"
MODEL_RERANK   = PROJECT_ROOT / "models" / "reranker"

EMBED_MODEL_NAME  = "intfloat/multilingual-e5-large-instruct"
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"       # multilingual, hỗ trợ tiếng Việt tốt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Retrieval params
BM25_TOP_K    = 50   # BM25 candidates
DENSE_TOP_K   = 50   # ChromaDB candidates
RRF_K         = 60   # Reciprocal Rank Fusion constant
RERANK_TOP_N  = 20   # Số candidates đưa vào reranker
FINAL_TOP_K   = 5    # Kết quả cuối

# Embedding instructions (asymmetric retrieval)
INSTRUCTIONS = {
    "legal"   : "Instruct: Tim dieu khoan phap luat Viet Nam lien quan.\nQuery: ",
    "forms"   : "Instruct: Tim mau bieu hanh chinh phu hop voi nhu cau soan thao.\nQuery: ",
    "examples": "Instruct: Tim vi du van ban hanh chinh tuong tu tinh huong sau.\nQuery: ",
}

# Tạo thư mục
BM25_DIR.mkdir(parents=True, exist_ok=True)
MODEL_EMBED.mkdir(parents=True, exist_ok=True)
MODEL_RERANK.mkdir(parents=True, exist_ok=True)

# Sanity checks
for name, path in {
    "legal_chunks"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms_chunks"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples_chunks": CHUNK_DIR / "examples_chunks.parquet",
    "chromadb"       : CHROMA_DIR,
}.items():
    status = "✅" if path.exists() else "❌ NOT FOUND"
    print(f"  {status} {name}: {path}")

## 2. Text Normalization Utilities

In [ ]:
def remove_diacritics(text: str) -> str:
    """Bỏ dấu tiếng Việt: dùng cho BM25 tokenization.
    Cả corpus lẫn query đều được normalize → đảm bảo match dù user gõ có/không dấu.
    """
    # Chuyển sang NFD (tách base char + combining marks)
    nfd = unicodedata.normalize("NFD", text)
    # Loại combining diacritical marks (category Mn)
    stripped = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    # Xử lý đ/Đ (không phải diacritics mà là ký tự riêng)
    stripped = stripped.replace("đ", "d").replace("Đ", "D")
    return stripped


def normalize_text(text: str) -> str:
    """Clean whitespace, lowercase."""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize_for_bm25(text: str) -> List[str]:
    """Tokenize text cho BM25: lowercase → bỏ dấu → tách word."""
    text = normalize_text(text)
    text = remove_diacritics(text)
    # Tách theo space + loại token rỗng, số, ký tự đặc biệt ngắn
    tokens = re.findall(r"[a-z][a-z0-9]{1,}", text)  # ít nhất 2 ký tự
    return tokens


# Quick test
test_cases = [
    "Quy định về hòa giải ở cơ sở",
    "quy dinh ve hoa giai o co so",      # không dấu
    "điều kiện để được cấp giấy phép kinh doanh",
    "dieu kien de duoc cap giay phep kinh doanh",
]
print("Tokenization test:")
for t in test_cases:
    tokens = tokenize_for_bm25(t)
    print(f"  Input : {t[:60]}")
    print(f"  Tokens: {tokens[:10]}")
    print()

## 3. Build & Persist BM25 Indexes

In [ ]:
def build_bm25_index(
    df: pd.DataFrame,
    text_col: str,
    save_path: Path,
    name: str,
    force_rebuild: bool = False,
) -> Tuple[BM25Okapi, List[str]]:
    """
    Build BM25 index từ DataFrame, persist bằng pickle.
    Nếu file đã tồn tại và force_rebuild=False → load từ disk.
    
    Returns: (bm25_index, chunk_ids)
    """
    meta_path = save_path.with_suffix(".meta.pkl")

    if save_path.exists() and meta_path.exists() and not force_rebuild:
        print(f"[{name}] Loading BM25 index from cache: {save_path}")
        t0 = time.time()
        with open(save_path, "rb") as f:
            bm25 = pickle.load(f)
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        print(f"  Loaded {len(meta['chunk_ids']):,} docs in {time.time()-t0:.1f}s")
        return bm25, meta["chunk_ids"]

    print(f"[{name}] Building BM25 index ({len(df):,} docs)...")
    t0 = time.time()

    chunk_ids = df["chunk_id"].tolist()
    corpus_tokens = [
        tokenize_for_bm25(text)
        for text in tqdm(df[text_col], desc=f"  Tokenizing {name}", leave=False)
    ]

    print(f"  Tokenized in {time.time()-t0:.1f}s, building BM25...")
    t1 = time.time()
    bm25 = BM25Okapi(corpus_tokens)
    print(f"  BM25 built in {time.time()-t1:.1f}s")

    # Persist
    with open(save_path, "wb") as f:
        pickle.dump(bm25, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(meta_path, "wb") as f:
        pickle.dump({"chunk_ids": chunk_ids}, f, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = save_path.stat().st_size / 1024 / 1024
    print(f"  Saved to {save_path} ({size_mb:.1f} MB) | Total: {time.time()-t0:.1f}s")
    return bm25, chunk_ids


print("BM25 builder defined.")

In [ ]:
# ============================================================
# Load parquet files & build BM25 indexes
# ============================================================

FORCE_REBUILD = False   # True để rebuild dù đã có cache

print("Loading chunk files...")
df_legal    = pd.read_parquet(CHUNK_DIR / "legal_chunks.parquet")
df_forms    = pd.read_parquet(CHUNK_DIR / "forms_chunks.parquet")
df_examples = pd.read_parquet(CHUNK_DIR / "examples_chunks.parquet")

print(f"  legal    : {len(df_legal):,} chunks")
print(f"  forms    : {len(df_forms):,} chunks")
print(f"  examples : {len(df_examples):,} chunks")
print()

# Build / load BM25 indexes
bm25_legal, ids_legal = build_bm25_index(
    df_legal, "text",
    BM25_DIR / "bm25_legal.pkl",
    "legal", force_rebuild=FORCE_REBUILD
)

bm25_forms, ids_forms = build_bm25_index(
    df_forms, "text",
    BM25_DIR / "bm25_forms.pkl",
    "forms", force_rebuild=FORCE_REBUILD
)

bm25_examples, ids_examples = build_bm25_index(
    df_examples, "text",
    BM25_DIR / "bm25_examples.pkl",
    "examples", force_rebuild=FORCE_REBUILD
)

print("\n✅ All BM25 indexes ready.")

# Build lookup: chunk_id → row metadata (for result display)
legal_meta    = df_legal.set_index("chunk_id").to_dict(orient="index")
forms_meta    = df_forms.set_index("chunk_id").to_dict(orient="index")
examples_meta = df_examples.set_index("chunk_id").to_dict(orient="index")

print("Metadata lookups built.")

## 4. Load Embedding Model & ChromaDB

In [ ]:
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
print("(~1.1 GB, sẽ cache sau lần đầu)")
t0 = time.time()

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_EMBED),
)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"✅ Loaded in {time.time()-t0:.1f}s | dim={EMBED_DIM} | device={DEVICE}")

In [ ]:
print(f"Connecting to ChromaDB: {CHROMA_DIR}")

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

col_legal    = chroma_client.get_collection("legal_chunks")
col_forms    = chroma_client.get_collection("forms_chunks")
col_examples = chroma_client.get_collection("examples_chunks")

print(f"✅ Collections loaded:")
print(f"   legal    : {col_legal.count():,}")
print(f"   forms    : {col_forms.count():,}")
print(f"   examples : {col_examples.count():,}")

## 5. Load Reranker

In [ ]:
print(f"Loading reranker: {RERANK_MODEL_NAME}")
print("(~1.1 GB, multilingual — hỗ trợ tiếng Việt)")
t0 = time.time()

reranker = CrossEncoder(
    RERANK_MODEL_NAME,
    device=DEVICE,
    cache_dir=str(MODEL_RERANK),
    max_length=512,
)
print(f"✅ Reranker loaded in {time.time()-t0:.1f}s")

## 6. Hybrid Retriever Class

In [ ]:
class HybridRetriever:
    """
    Hybrid retrieval: BM25 (sparse) + ChromaDB (dense) → RRF → Cross-Encoder Reranker.
    
    Hỗ trợ cả query có dấu lẫn không dấu:
    - BM25  : normalize về không dấu → match keyword
    - Dense : dùng text gốc (embedding model tự handle)
    """

    def __init__(
        self,
        bm25_index: BM25Okapi,
        bm25_ids: List[str],
        chroma_col,
        meta_lookup: Dict,
        embed_model: SentenceTransformer,
        reranker: CrossEncoder,
        instruction: str,
        collection_name: str,
        bm25_top_k: int = BM25_TOP_K,
        dense_top_k: int = DENSE_TOP_K,
        rrf_k: int = RRF_K,
        rerank_top_n: int = RERANK_TOP_N,
        final_top_k: int = FINAL_TOP_K,
    ):
        self.bm25        = bm25_index
        self.bm25_ids    = bm25_ids
        self.col         = chroma_col
        self.meta        = meta_lookup
        self.embed_model = embed_model
        self.reranker    = reranker
        self.instruction = instruction
        self.name        = collection_name
        self.bm25_top_k  = bm25_top_k
        self.dense_top_k = dense_top_k
        self.rrf_k       = rrf_k
        self.rerank_top_n = rerank_top_n
        self.final_top_k = final_top_k

    # ----------------------------------------------------------
    # BM25 search
    # ----------------------------------------------------------
    def _bm25_search(self, query: str) -> List[Tuple[str, float]]:
        """Returns list of (chunk_id, bm25_score), sorted desc."""
        tokens = tokenize_for_bm25(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][: self.bm25_top_k]
        # Lọc score = 0 (không có overlap)
        return [
            (self.bm25_ids[i], float(scores[i]))
            for i in top_idx
            if scores[i] > 0
        ]

    # ----------------------------------------------------------
    # Dense (ChromaDB) search
    # ----------------------------------------------------------
    def _dense_search(
        self, query: str, where: Optional[Dict] = None
    ) -> List[Tuple[str, float, str]]:
        """Returns list of (chunk_id, distance, document_text), sorted asc distance."""
        # Asymmetric query encoding
        q_text = self.instruction + query
        q_vec  = self.embed_model.encode(
            [q_text],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]

        kwargs = dict(
            query_embeddings=[q_vec.tolist()],
            n_results=self.dense_top_k,
            include=["documents", "distances"],
        )
        if where:
            kwargs["where"] = where

        res = self.col.query(**kwargs)
        results = []
        for cid, dist, doc in zip(
            res["ids"][0], res["distances"][0], res["documents"][0]
        ):
            results.append((cid, float(dist), doc))
        return results   # sorted by dist asc (best first)

    # ----------------------------------------------------------
    # Reciprocal Rank Fusion
    # ----------------------------------------------------------
    def _rrf(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float, str]],
    ) -> List[Tuple[str, float]]:
        """
        Merge hai ranked list bằng RRF.
        Returns list of (chunk_id, rrf_score) sorted desc.
        """
        scores: Dict[str, float] = {}

        for rank, (cid, _) in enumerate(bm25_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)

        for rank, (cid, _, _) in enumerate(dense_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)

        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # ----------------------------------------------------------
    # Cross-Encoder Reranker
    # ----------------------------------------------------------
    def _rerank(
        self,
        query: str,
        candidate_ids: List[str],
        doc_texts: Dict[str, str],
    ) -> List[Tuple[str, float]]:
        """
        Rerank top-N candidates với cross-encoder.
        Returns list of (chunk_id, rerank_score) sorted desc.
        """
        pairs = [(query, doc_texts[cid]) for cid in candidate_ids if cid in doc_texts]
        if not pairs:
            return [(cid, 0.0) for cid in candidate_ids]

        rerank_scores = self.reranker.predict(pairs, show_progress_bar=False)
        ranked = sorted(
            zip(candidate_ids[: len(pairs)], rerank_scores.tolist()),
            key=lambda x: x[1],
            reverse=True,
        )
        return ranked

    # ----------------------------------------------------------
    # Public API
    # ----------------------------------------------------------
    def retrieve(
        self,
        query: str,
        where: Optional[Dict] = None,
        use_reranker: bool = True,
        return_scores: bool = False,
    ) -> List[Dict]:
        """
        Full hybrid retrieval pipeline.

        Args:
            query        : User query (có dấu hoặc không dấu đều OK)
            where        : ChromaDB metadata filter (VD: {"type_normalized": "NGHỊ ĐỊNH"})
            use_reranker : False để bỏ qua reranker (nhanh hơn)
            return_scores: True để trả về dict đầy đủ scores

        Returns:
            List[Dict] với final_top_k kết quả, mỗi dict có:
              chunk_id, text, metadata, bm25_score, dense_dist, rrf_score, rerank_score
        """
        t0 = time.time()

        # 1. BM25 search
        bm25_res  = self._bm25_search(query)

        # 2. Dense search
        dense_res = self._dense_search(query, where=where)

        # 3. Build doc text lookup (chunk_id → text)
        doc_texts: Dict[str, str] = {}
        for cid, _, doc in dense_res:
            doc_texts[cid] = doc
        # Thêm text từ BM25 hits (lấy từ meta_lookup)
        for cid, _ in bm25_res:
            if cid not in doc_texts and cid in self.meta:
                doc_texts[cid] = self.meta[cid].get("text", "")

        # Score lookups
        bm25_score_map  = {cid: s for cid, s in bm25_res}
        dense_dist_map  = {cid: d for cid, d, _ in dense_res}

        # 4. RRF merge
        rrf_ranked = self._rrf(bm25_res, dense_res)

        # 5. Take top-N for reranker
        top_n_ids = [cid for cid, _ in rrf_ranked[: self.rerank_top_n]]
        rrf_score_map = {cid: s for cid, s in rrf_ranked}

        # 6. Rerank
        if use_reranker and top_n_ids:
            final_ranked = self._rerank(query, top_n_ids, doc_texts)
        else:
            final_ranked = [(cid, rrf_score_map.get(cid, 0.0)) for cid in top_n_ids]

        # 7. Build output
        results = []
        for cid, rerank_score in final_ranked[: self.final_top_k]:
            meta = self.meta.get(cid, {})
            results.append({
                "chunk_id"     : cid,
                "text"         : doc_texts.get(cid, meta.get("text", "")),
                "metadata"     : meta,
                "bm25_score"   : round(bm25_score_map.get(cid, 0.0), 4),
                "dense_dist"   : round(dense_dist_map.get(cid, 1.0), 4),
                "rrf_score"    : round(rrf_score_map.get(cid, 0.0), 6),
                "rerank_score" : round(float(rerank_score), 4),
                "latency_ms"   : round((time.time() - t0) * 1000, 1),
            })

        return results

    def retrieve_only_semantic(
        self,
        query: str,
        where: Optional[Dict] = None,
    ) -> List[Dict]:
        """Chỉ dùng ChromaDB (để so sánh baseline)."""
        dense_res = self._dense_search(query, where=where)
        results = []
        for cid, dist, doc in dense_res[: self.final_top_k]:
            meta = self.meta.get(cid, {})
            results.append({
                "chunk_id"  : cid,
                "text"      : doc,
                "metadata"  : meta,
                "dense_dist": round(dist, 4),
            })
        return results


print("HybridRetriever class defined.")

## 7. Khởi tạo Retrievers

In [ ]:
retriever_legal = HybridRetriever(
    bm25_index      = bm25_legal,
    bm25_ids        = ids_legal,
    chroma_col      = col_legal,
    meta_lookup     = legal_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["legal"],
    collection_name = "legal",
)

retriever_forms = HybridRetriever(
    bm25_index      = bm25_forms,
    bm25_ids        = ids_forms,
    chroma_col      = col_forms,
    meta_lookup     = forms_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["forms"],
    collection_name = "forms",
    final_top_k     = 3,
)

retriever_examples = HybridRetriever(
    bm25_index      = bm25_examples,
    bm25_ids        = ids_examples,
    chroma_col      = col_examples,
    meta_lookup     = examples_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["examples"],
    collection_name = "examples",
    final_top_k     = 3,
)

print("✅ All retrievers ready.")

## 8. Display Helper

In [ ]:
def print_legal_results(results: List[Dict], query: str, mode: str = "hybrid"):
    print(f"\n{'='*70}")
    print(f"[{mode.upper()}] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        doc_no   = meta.get("source_doc_no", meta.get("id", "?"))
        article  = meta.get("article", "")
        doc_type = meta.get("type_normalized", "")
        text_preview = r["text"][:200].replace("\n", " ")
        score_str = (
            f"rerank={r.get('rerank_score','N/A'):.4f} | "
            f"rrf={r.get('rrf_score',0):.5f} | "
            f"bm25={r.get('bm25_score',0):.2f} | "
            f"dense_dist={r.get('dense_dist',1):.4f}"
        ) if "rerank_score" in r else f"dense_dist={r.get('dense_dist',1):.4f}"

        print(f"\n  [{i}] {doc_no} | {doc_type}")
        print(f"      📌 {article}")
        print(f"      🎯 {score_str}")
        print(f"      📝 {text_preview}...")
    latency = results[0].get("latency_ms", 0) if results else 0
    print(f"\n  ⏱  Latency: {latency}ms")


def print_forms_results(results: List[Dict], query: str, mode: str = "hybrid"):
    print(f"\n{'='*70}")
    print(f"[{mode.upper()} - FORMS] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        form_id   = meta.get("form_id", "?")
        form_type = meta.get("form_type", "")
        purpose   = meta.get("purpose", "")
        score_str = f"rerank={r.get('rerank_score',0):.4f}" if "rerank_score" in r else f"dist={r.get('dense_dist',1):.4f}"
        print(f"  [{i}] {form_id} | {form_type} | {score_str}")
        print(f"      Purpose: {purpose[:200]}")


def print_examples_results(results: List[Dict], query: str, mode: str = "hybrid"):
    print(f"\n{'='*70}")
    print(f"[{mode.upper()} - EXAMPLES] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        form_id  = meta.get("form_id", "?")
        scenario = meta.get("scenario", "")
        text_preview = r["text"][:150].replace("\n", " ")
        score_str = f"rerank={r.get('rerank_score',0):.4f}" if "rerank_score" in r else f"dist={r.get('dense_dist',1):.4f}"
        print(f"  [{i}] {form_id} | {score_str}")
        print(f"      Scenario: {scenario[:200]}")
        print(f"      Preview : {text_preview}...")


print("Display helpers defined.")

## 9. Test Queries — Legal Retrieval

In [ ]:
# ============================================================
# TEST LEGAL — Query đa dạng: có dấu, không dấu, câu hỏi tự nhiên
# ============================================================

test_legal_queries = [
    # Trường hợp từ context (baseline đã test)
    "quy dinh ve hoa giai o co so",
    "dieu kien de duoc cap giay phep kinh doanh",
    "trach nhiem cua Bo truong trong ban hanh van ban",

    # Có dấu
    "Quy định về hòa giải ở cơ sở",
    "Điều kiện thành lập doanh nghiệp",
    "Quyền và nghĩa vụ của người lao động",

    # Câu hỏi tự nhiên
    "thủ tục xin cấp phép xây dựng nhà ở",
    "xử phạt vi phạm hành chính trong lĩnh vực giao thông",
    "quy trình bổ nhiệm cán bộ công chức",
    "bảo hiểm xã hội bắt buộc cho người lao động",
]

for query in test_legal_queries:
    # Hybrid
    results_hybrid = retriever_legal.retrieve(query, use_reranker=True)
    print_legal_results(results_hybrid, query, mode="hybrid")

    # Baseline: semantic only (để so sánh)
    results_semantic = retriever_legal.retrieve_only_semantic(query)
    print_legal_results(results_semantic, query, mode="semantic-only")
    print("\n" + "-"*70)

## 10. Test Queries — Forms & Examples

In [ ]:
# ============================================================
# TEST FORMS & EXAMPLES
# ============================================================

test_form_queries = [
    "soạn thảo công văn gửi sở tài chính",
    "viết giấy mời họp ban lãnh đạo",
    "làm tờ trình đề nghị phê duyệt dự án",
    "giấy giới thiệu cán bộ đi công tác",
    "quyết định bổ nhiệm trưởng phòng",
    "báo cáo tình hình thực hiện nhiệm vụ",
]

for query in test_form_queries:
    res_forms    = retriever_forms.retrieve(query, use_reranker=True)
    res_examples = retriever_examples.retrieve(query, use_reranker=True)
    print_forms_results(res_forms, query)
    print_examples_results(res_examples, query)

## 11. Test với Metadata Filter

In [ ]:
# ============================================================
# TEST FILTER — lọc theo loại văn bản hoặc số hiệu
# ============================================================

filter_tests = [
    {
        "query" : "điều kiện kinh doanh dịch vụ logistics",
        "where" : {"type_normalized": "NGHỊ ĐỊNH"},
        "desc"  : "Chỉ tìm trong NGHỊ ĐỊNH",
    },
    {
        "query" : "quyền của người lao động",
        "where" : {"type_normalized": "LUẬT"},
        "desc"  : "Chỉ tìm trong LUẬT",
    },
    {
        "query" : "xử phạt vi phạm",
        "where" : {"type_normalized": "THÔNG TƯ"},
        "desc"  : "Chỉ tìm trong THÔNG TƯ",
    },
]

for test in filter_tests:
    print(f"\n>>> Filter: {test['desc']}")
    results = retriever_legal.retrieve(
        test["query"],
        where=test["where"],
        use_reranker=True,
    )
    print_legal_results(results, test["query"], mode=f"hybrid+filter")

## 12. Benchmark: Hybrid vs Semantic-Only

In [ ]:
# ============================================================
# BENCHMARK — So sánh chất lượng trước/sau hybrid
# Dùng BM25 score > 0 ở top-1 làm tín hiệu keyword match
# ============================================================

benchmark_queries = [
    "quy dinh ve hoa giai o co so",
    "dieu kien de duoc cap giay phep kinh doanh",
    "trach nhiem cua Bo truong trong ban hanh van ban",
    "xu phat vi pham hanh chinh giao thong duong bo",
    "quyen va nghia vu nguoi su dung dat",
    "thu tuc dang ky kinh doanh ho ca the",
    "chinh sach ho tro nguoi co cong voi cach mang",
    "quy trinh kiem tra chat luong hang hoa xuat nhap khau",
]

print(f"{'Query':<55} {'Semantic top1 doc':^25} {'Hybrid top1 doc':^25}")
print("-" * 110)

for q in benchmark_queries:
    sem = retriever_legal.retrieve_only_semantic(q)
    hyb = retriever_legal.retrieve(q, use_reranker=True)

    sem_doc = sem[0]["metadata"].get("source_doc_no", "?")[:22] if sem else "N/A"
    hyb_doc = hyb[0]["metadata"].get("source_doc_no", "?")[:22] if hyb else "N/A"
    hyb_rerank = hyb[0].get("rerank_score", 0) if hyb else 0
    hyb_bm25   = hyb[0].get("bm25_score", 0)   if hyb else 0

    same = "=" if sem_doc == hyb_doc else "≠"
    print(f"{q[:54]:<55} {sem_doc:<25} {hyb_doc:<25} {same} rerank={hyb_rerank:.3f} bm25={hyb_bm25:.1f}")

print()
print("Chú thích: '≠' = hybrid trả về kết quả khác semantic → kiểm tra xem cái nào đúng hơn")

## 13. Convenience Functions — Dùng trong RAG Pipeline

In [ ]:
# ============================================================
# PUBLIC API — dùng trong bước tiếp theo (RAG generation)
# ============================================================

def retrieve_legal(
    query: str,
    top_k: int = 5,
    where: Optional[Dict] = None,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Truy xuất điều khoản pháp luật liên quan.
    
    Example:
        results = retrieve_legal("điều kiện thành lập doanh nghiệp")
        results = retrieve_legal("quy định", where={"type_normalized": "NGHỊ ĐỊNH"})
    """
    retriever_legal.final_top_k = top_k
    return retriever_legal.retrieve(query, where=where, use_reranker=use_reranker)


def retrieve_forms(
    query: str,
    top_k: int = 3,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Truy xuất biểu mẫu hành chính phù hợp.
    
    Example:
        results = retrieve_forms("soạn thảo công văn xin phép")
    """
    retriever_forms.final_top_k = top_k
    return retriever_forms.retrieve(query, use_reranker=use_reranker)


def retrieve_examples(
    query: str,
    top_k: int = 3,
    form_id: Optional[str] = None,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Truy xuất ví dụ văn bản tương tự (few-shot).
    
    Example:
        results = retrieve_examples("giấy mời họp", form_id="Form_07")
    """
    retriever_examples.final_top_k = top_k
    where = {"form_id": form_id} if form_id else None
    return retriever_examples.retrieve(query, where=where, use_reranker=use_reranker)


def retrieve_all(
    query: str,
    legal_top_k: int = 5,
    forms_top_k: int = 3,
    examples_top_k: int = 3,
    legal_filter: Optional[Dict] = None,
) -> Dict[str, List[Dict]]:
    """
    Chạy cả 3 retrievers cùng lúc. Dùng trong RAG pipeline.
    
    Returns:
        {"legal": [...], "forms": [...], "examples": [...]}
    """
    return {
        "legal"   : retrieve_legal(query, top_k=legal_top_k, where=legal_filter),
        "forms"   : retrieve_forms(query, top_k=forms_top_k),
        "examples": retrieve_examples(query, top_k=examples_top_k),
    }


# ── Quick demo ───────────────────────────────────────────────
print("Demo: retrieve_all()")
demo_query = "soạn thảo quyết định bổ nhiệm cán bộ"
all_results = retrieve_all(demo_query)

print(f"\nQuery: '{demo_query}'")
print(f"  Legal    : {len(all_results['legal'])} kết quả")
print(f"  Forms    : {len(all_results['forms'])} kết quả")
print(f"  Examples : {len(all_results['examples'])} kết quả")

print("\nTop legal result:")
if all_results["legal"]:
    r = all_results["legal"][0]
    print(f"  {r['metadata'].get('source_doc_no','?')} | {r['metadata'].get('article','')}")
    print(f"  rerank={r['rerank_score']:.4f} | {r['text']}...")

---

## Summary

| Component | Model / Library | Mô tả |
|---|---|---|
| **BM25** | `rank_bm25.BM25Okapi` | Keyword match, normalize không dấu, persist pkl |
| **Dense** | `multilingual-e5-large-instruct` | Asymmetric semantic search, L2-normalized |
| **Fusion** | Reciprocal Rank Fusion (k=60) | Merge BM25 + dense ranked lists |
| **Reranker** | `BAAI/bge-reranker-v2-m3` | Cross-encoder, multilingual, tiếng Việt tốt |

**Bước tiếp theo:** Dùng `retrieve_all()` trong RAG generation pipeline.